[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/11_random_forest/01_bagging_and_variance_reduction.ipynb)

# Bài 01 — Bagging: giảm variance bằng cách lấy trung bình

**🎯 Mục tiêu**
- Hiểu vì sao **trung bình nhiều model** giảm variance — và giới hạn do **tương quan** giữa các model: $\rho\sigma^2 + \frac{1 - \rho}{B}\sigma^2$.
- **Bootstrap**: lấy mẫu có hoàn lại; ~63.2% điểm khác nhau mỗi mẫu.
- **Bagging** (Bootstrap AGGregatING) cho cây quyết định: ranh giới mượt hơn, ổn định hơn.
- Đo bias–variance bằng mô phỏng: bagging giảm variance, gần như không đổi bias; chỉ giúp model **không ổn định**.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/11_random_forest

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.ensemble import BaggingClassifier, BaggingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

from ml_course.datasets import make_classification_2d
from ml_course.visualization import plot_decision_boundary, set_style

set_style()
rng = np.random.default_rng(0)

## 1. Trí tuệ đám đông — và giới hạn của nó

$B$ model, mỗi model có dự đoán là biến ngẫu nhiên với phương sai $\sigma^2$, tương quan đôi một $\rho$. Phương sai của **trung bình**:

$$\operatorname{Var}\left(\frac1B\sum_{b=1}^B f_b\right) = \rho\,\sigma^2 + \frac{1 - \rho}{B}\,\sigma^2$$

- Model độc lập ($\rho = 0$): variance giảm $B$ lần.
- Model tương quan: dù $B \to \infty$, variance không xuống dưới $\rho\sigma^2$. → Muốn ensemble hiệu quả, các model phải **khác nhau** (đa dạng). Đây chính là ý tưởng của Random Forest (bài 02).

In [ ]:
sigma = 1.0
Bs = np.arange(1, 101)
fig, ax = plt.subplots(figsize=(8, 4))
for rho in [0.0, 0.2, 0.5, 0.8]:
    ax.plot(Bs, rho * sigma**2 + (1 - rho) * sigma**2 / Bs, label=f"ρ = {rho} (lý thuyết)")
    # kiểm chứng bằng mô phỏng: B biến chuẩn có tương quan ρ
    cov = rho * np.ones((100, 100)) + (1 - rho) * np.eye(100)
    samples = rng.multivariate_normal(np.zeros(100), cov, size=4000)
    for B in [5, 25, 100]:
        ax.plot(B, samples[:, :B].mean(1).var(), "ko", ms=4)
ax.set_xlabel("số model B")
ax.set_ylabel("variance của trung bình")
ax.set_title("Chấm đen: mô phỏng — khớp công thức")
ax.legend()
plt.show()

## 2. Bootstrap

Từ tập train $n$ mẫu, rút **có hoàn lại** $n$ mẫu → một "tập dữ liệu mới" hơi khác. Xác suất một điểm **không** được chọn lần nào: $(1 - 1/n)^n \to e^{-1} \approx 0.368$. Vậy mỗi mẫu bootstrap chứa ~63.2% điểm khác nhau; ~36.8% còn lại gọi là **out-of-bag (OOB)** — dùng để đánh giá miễn phí (bài 03).

In [ ]:
for n in [10, 100, 1000, 100_000]:
    fracs = [len(np.unique(rng.integers(0, n, n))) / n for _ in range(200)]
    print(f"n = {n:6d}: tỉ lệ điểm khác nhau trong mẫu bootstrap = {np.mean(fracs):.4f} | lý thuyết 1 − (1 − 1/n)^n = {1 - (1 - 1 / n) ** n:.4f}")
print(f"giới hạn 1 − 1/e = {1 - np.exp(-1):.4f}")

## 3. Bagging cây quyết định

1. Tạo $B$ mẫu bootstrap.
2. Train một cây **sâu** (bias thấp, variance cao) trên mỗi mẫu.
3. Dự đoán: **bỏ phiếu** / trung bình xác suất (phân loại) hoặc **trung bình** (hồi quy).

In [ ]:
def bagging_fit(X, y, n_estimators=100, seed=0):
    r = np.random.default_rng(seed)
    trees = []
    for _ in range(n_estimators):
        idx = r.integers(0, len(X), len(X))                        # bootstrap
        trees.append(DecisionTreeClassifier(random_state=0).fit(X[idx], y[idx]))
    return trees


def bagging_predict_proba(trees, X):
    return np.mean([t.predict_proba(X) for t in trees], axis=0)    # trung bình xác suất (soft voting)


class BaggedTrees:                                                 # bọc lại để vẽ decision boundary
    def __init__(self, trees):
        self.trees = trees

    def predict(self, X):
        return np.argmax(bagging_predict_proba(self.trees, X), axis=1)


X, y = make_classification_2d("moons", n=300, noise=0.35, seed=1)
X_test, y_test = make_classification_2d("moons", n=3000, noise=0.35, seed=2)
single = DecisionTreeClassifier(random_state=0).fit(X, y)
bagged = BaggedTrees(bagging_fit(X, y, n_estimators=200))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
plot_decision_boundary(single, X, y, ax=axes[0])
axes[0].set_title(f"1 cây sâu: test acc = {single.score(X_test, y_test):.3f}")
plot_decision_boundary(bagged, X, y, ax=axes[1])
axes[1].set_title(f"Bagging 200 cây: test acc = {np.mean(bagged.predict(X_test) == y_test):.3f}")
plt.show()

In [ ]:
trees = bagged.trees
accs = [np.mean(np.argmax(bagging_predict_proba(trees[:B], X_test), 1) == y_test) for B in range(1, 201)]
plt.plot(range(1, 201), accs)
plt.axhline(single.score(X_test, y_test), color="gray", ls="--", label="1 cây")
plt.xlabel("số cây B")
plt.ylabel("test accuracy")
plt.title("Thêm cây: accuracy tăng nhanh rồi bão hoà — không bao giờ overfit vì thêm cây")
plt.legend()
plt.show()

> 💡 Tăng $B$ chỉ làm ước lượng trung bình **chính xác hơn** — không làm model phức tạp hơn → **thêm cây không gây overfit** (chỉ tốn thời gian). Đây là khác biệt lớn với boosting (chương 12).

## 4. Bias–variance: bagging giảm cái gì?

Mô phỏng: lặp lại 100 lần — sinh tập train mới từ cùng một quy luật, train model, dự đoán trên lưới cố định. Tại mỗi điểm $x$: **bias²** = (trung bình dự đoán − hàm thật)², **variance** = phương sai dự đoán qua 100 lần.

In [ ]:
def f_true(x):
    return np.sin(2 * x) + 0.3 * x


grid = np.linspace(-3, 3, 200).reshape(-1, 1)


def bias_variance(make_model, n_runs=100, n=80, noise=0.4):
    preds = []
    for run in range(n_runs):
        r = np.random.default_rng(run)
        x = r.uniform(-3, 3, (n, 1))
        yy = f_true(x.ravel()) + r.normal(0, noise, n)
        preds.append(make_model().fit(x, yy).predict(grid))
    preds = np.array(preds)
    bias2 = np.mean((preds.mean(0) - f_true(grid.ravel())) ** 2)
    variance = np.mean(preds.var(0))
    return bias2, variance


models = {
    "cây sâu": lambda: DecisionTreeRegressor(random_state=0),
    "bagging 50 cây sâu": lambda: BaggingRegressor(DecisionTreeRegressor(), n_estimators=50, random_state=0),
    "cây depth 3": lambda: DecisionTreeRegressor(max_depth=3, random_state=0),
    "bagging 50 cây depth 3": lambda: BaggingRegressor(DecisionTreeRegressor(max_depth=3), n_estimators=50, random_state=0),
    "Linear Regression": lambda: LinearRegression(),
    "bagging 50 Linear Regression": lambda: BaggingRegressor(LinearRegression(), n_estimators=50, random_state=0),
}
for name, make in models.items():
    b2, v = bias_variance(make)
    print(f"{name:30s} bias² = {b2:.4f} | variance = {v:.4f} | tổng = {b2 + v:.4f}")

Đọc kết quả:
- **Bias gần như không đổi** trong mọi trường hợp — bagging không làm model "thông minh" hơn.
- **Cây sâu**: variance giảm khoảng một nửa → tổng lỗi giảm mạnh. Nhưng không giảm được nhiều hơn: 50 cây bootstrap từ **cùng** 80 điểm vẫn **tương quan** với nhau (công thức mục 1: sàn $\rho\sigma^2$).
- **Cây depth 3**: variance cũng giảm một nửa; ở đây (rất ít dữ liệu, nhiễu lớn) bagging cây nông còn cho tổng lỗi **thấp nhất** — độ sâu/kích thước lá vẫn là hyperparameter cần chọn.
- **Linear Regression**: model ổn định (variance vốn rất thấp) → bagging **vô ích**; bias (model tuyến tính không bắt được hình sin) giữ nguyên.

→ Bagging giúp nhiều nhất cho base learner **không ổn định** (variance cao) như cây quyết định; để giảm variance hơn nữa phải giảm **tương quan** giữa các cây → Random Forest (bài 02).

## 5. Với scikit-learn

In [ ]:
for n_est in [1, 10, 100]:
    bag = BaggingClassifier(DecisionTreeClassifier(), n_estimators=n_est, random_state=0).fit(X, y)
    print(f"BaggingClassifier, {n_est:3d} cây: test acc = {bag.score(X_test, y_test):.3f}")

## 🧠 Kiểm tra nhanh

1. 50 model, mỗi model variance 1, tương quan 0.3. Variance của trung bình? Khi số model → ∞?
2. Vì sao mỗi mẫu bootstrap chỉ chứa khoảng 63% số điểm khác nhau?
3. Thêm cây vào bagging có làm overfit không?
4. Vì sao bagging Linear Regression gần như không có tác dụng?

## 📌 Tóm tắt

- Trung bình giảm variance; tương quan $\rho$ đặt "sàn" $\rho\sigma^2$.
- Bootstrap tạo đa dạng; ~36.8% điểm OOB mỗi mẫu.
- Bagging cây sâu: giảm variance mạnh, bias gần như không đổi, thêm cây không overfit.

## 📚 Tài liệu tham khảo

- Breiman (1996) — *Bagging Predictors*
- Efron & Tibshirani (1993) — *An Introduction to the Bootstrap*
- ESL §8.7, §15.2